In [1]:
import sys
sys.path.insert(0, "/projappl/project_2012747/mars/MarS")  # folder that contains market_simulation/
from market_simulation.models.order_model import OrderModel

2026-01-31 16:16:24,332 - /projappl/project_2012747/mars/MarS/market_simulation/__init__.py:15 - INFO - init logging


/PUHTI_TYKKY_Quvj2Tb/miniforge/envs/env1/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import os


sweep_configuration = {
    "method": "grid",
    "metric": {"goal": "minimize", "name": "val/loss"},
    "parameters": {
        "train_fraction": {"values": [1.0, 0.5]},
        "model_variant": {"values": ["base", "small"]},

        # keep a few training knobs configurable if you want
        "lr": {"value": 3e-4},
        "batch_size": {"value": 8},
        "max_steps": {"value": 20000},
        "eval_every": {"value": 100},
        "val_max_batches": {"value": 200},
        "seed": {"value": 123},
    },
}


In [3]:
import glob, bisect
import os, zipfile
import numpy as np
import zarr, torch
import random
import torch.nn.functional as F
from tqdm import tqdm
from torch.utils.data import Dataset, IterableDataset, DataLoader, Subset
from datasets import load_dataset
from zarr.storage import DirectoryStore
from functools import lru_cache
from typing import Optional
from market_simulation.models.order_model import OrderModel







# -------------------------
# Fixed config (shared)
# -------------------------
K = 1024
batch_size = 4096
train_fraction=0.8
seed=42
TRAIN_STRIDE = 16
VAL_STRIDE = 16

USE_AMP = True
AMP_DTYPE = torch.bfloat16  # or torch.float16

torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

feat_cols = [f"f{i}" for i in range(15)]


def add_f4(batch):
    t = np.asarray(batch["Time"], dtype=np.int64)
    t_sec = (t // 1_000_000_000).astype(np.int64)
    batch["f4"] = np.clip(t_sec - 34200, 0, 23399).astype(np.int64)
    return batch


def load_parquet_as_cols(path: str):
    ds = load_dataset("parquet", data_files={"data": path})["data"]
    ds = ds.map(add_f4, batched=True, batch_size=200_000, num_proc=1)

    # keep only f0..f14
    drop_cols = [c for c in ds.column_names if c not in feat_cols]
    if drop_cols:
        ds = ds.remove_columns(drop_cols)

    ds = ds.with_format("numpy")
    cols = {c: ds[c] for c in feat_cols}
    return cols



def make_train_val_loaders(ds, val_frac=0.01, seed=0, **dl_kwargs):
    n = len(ds)
    idx = list(range(n))
    random.Random(seed).shuffle(idx)

    n_val = int(n * val_frac)
    val_idx = idx[:n_val]
    train_idx = idx[n_val:]

    train_dl = DataLoader(Subset(ds, train_idx), shuffle=True, **dl_kwargs)
    val_dl   = DataLoader(Subset(ds, val_idx),   shuffle=False, **dl_kwargs)
    return train_dl, val_dl


# 1) unzip *.zarr.zip -> folders (once)
def unzip_zarr_zips(train_dir="train", pattern="*.zarr.zip"):
    for zpath in sorted(glob.glob(os.path.join(train_dir, pattern))):
        out_dir = zpath[:-4]  # strip ".zip" -> "... .zarr"
        if os.path.isdir(out_dir) and os.listdir(out_dir):
            continue
        os.makedirs(out_dir, exist_ok=True)
        with zipfile.ZipFile(zpath) as zf:
            zf.extractall(out_dir)
    return sorted(glob.glob(os.path.join(train_dir, pattern.replace(".zip", ""))))  # *.zarr dirs


# 2) dataset over many DirectoryStores (fast)
class MultiDirZarrOrderDataset(Dataset):
    def __init__(self, zarr_dirs, seq_len=1024):
        self.seq_len = seq_len
        self.paths = list(zarr_dirs)

        self.lens = []
        for p in self.paths:
            X = zarr.open(DirectoryStore(p), path="X", mode="r")
            self.lens.append(X.shape[0] - seq_len - 1)

        self.cum = []
        s = 0
        for L in self.lens:
            s += max(0, L)
            self.cum.append(s)

    def __len__(self):
        return self.cum[-1] if self.cum else 0

    @staticmethod
    @lru_cache(maxsize=16)
    def _open_X(dir_path):
        store = DirectoryStore(dir_path)
        X = zarr.open(store=store, path="X", mode="r")
        return X

    def __getitem__(self, idx):
        fi = bisect.bisect_right(self.cum, idx)
        prev = 0 if fi == 0 else self.cum[fi - 1]
        j = idx - prev

        X = self._open_X(self.paths[fi])
        x = X[j : j + self.seq_len]      # (1024, 15)
        #y = X[j + self.seq_len, 0]       # next order index (f0)
        return torch.from_numpy(x).long() #, torch.tensor(y, dtype=torch.long)


def lm_loss_all_positions(logits: torch.Tensor, X: torch.Tensor) -> torch.Tensor:
    targets = X[:, :, 0]           # (B, K)
    logits_s = logits[:, :-1, :]   # (B, K-1, vocab)
    targ_s   = targets[:, 1:]      # (B, K-1)
    return F.cross_entropy(
        logits_s.reshape(-1, logits_s.size(-1)),
        targ_s.reshape(-1),
        reduction="mean",
    )


def pick_train_files(all_train_files, train_fraction: float, seed: int):
    """
    Implements "half the training set" as half the *files* (deterministic shuffle).
    """
    if train_fraction >= 0.999:
        return all_train_files

    if train_fraction <= 0.0:
        raise ValueError("train_fraction must be > 0")

    rng = np.random.default_rng(seed)
    files = list(all_train_files)
    rng.shuffle(files)
    n = max(1, int(round(len(files) * train_fraction)))
    return sorted(files[:n])


def build_model_from_variant(model_variant: str):
    """
    base ~ your current config (emb=64, layers=2, heads=4)
    small = fewer params (emb=48, layers=1, heads=4) -> significantly smaller
    """
    if model_variant == "base":
        EMB_DIM, NUM_LAYERS, NUM_HEADS = 64, 2, 4
    elif model_variant == "small":
        # smaller than base; keep heads dividing emb_dim nicely
        EMB_DIM, NUM_LAYERS, NUM_HEADS = 48, 1, 4
    else:
        raise ValueError(f"Unknown model_variant={model_variant}")

    model = OrderModel(
        emb_dim=EMB_DIM,
        num_layers=NUM_LAYERS,
        num_heads=NUM_HEADS,
        num_max_orders=K,
    ).to(device)

    return model, {"emb_dim": EMB_DIM, "num_layers": NUM_LAYERS, "num_heads": NUM_HEADS}


@torch.no_grad()
def compute_val_loss(model, val_dl, val_max_batches: Optional[int]):
    model.eval()
    total = 0.0
    count = 0

    for b, X in enumerate(val_dl, start=1):
        X = X.to(device, non_blocking=True)

        with torch.amp.autocast("cuda", enabled=(USE_AMP and device.type == "cuda"), dtype=AMP_DTYPE):
            logits = model(X)
            loss = lm_loss_all_positions(logits, X)

        n = X.size(0) * (X.size(1) - 1)
        total += loss.item() * n
        count += n

        if val_max_batches is not None and b >= val_max_batches:
            break

    return total / max(1, count)


def train_fn():

    model_variant = "base"
    lr = 3e-4
    max_steps=20000
    eval_every=100
    val_max_batches=200
    seed=123

    # reproducibility
    torch.manual_seed(int(seed))
    np.random.seed(int(seed))

    # # -------------------------
    # # Data selection (train_fraction)
    # # -------------------------
    # all_train_files = sorted(glob.glob("../data/features/train_*.parquet"))
    # val_files       = sorted(glob.glob("../data/features/test_*.parquet"))
    # train_files = pick_train_files(all_train_files, float(train_fraction), int(seed))
    # # Load segments
    # train_segments = [load_parquet_as_cols(p) for p in train_files]
    # val_segments   = [load_parquet_as_cols(p) for p in val_files]



    
    # 3) usage
    zarr_dirs = unzip_zarr_zips("../../data/order_model/train", "*_features.zarr.zip")  # creates train/*_features.zarr/
    ds = MultiDirZarrOrderDataset(zarr_dirs, seq_len=1024)
    
    train_dl, val_dl = make_train_val_loaders(
        ds,
        val_frac=0.01,
        seed=42,
        batch_size=4096,
        num_workers=8,
        pin_memory=True,
        drop_last=True,
    )

    
    
    # -------------------------
    # Model selection (model_variant)
    # -------------------------
    model, model_hps = build_model_from_variant(str(model_variant))
    n_params = sum(p.numel() for p in model.parameters())

    opt = torch.optim.AdamW(model.parameters(), lr=float(lr))

    use_scaler = (USE_AMP and device.type == "cuda" and AMP_DTYPE == torch.float16)
    scaler = torch.amp.GradScaler("cuda", enabled=use_scaler)

    # -------------------------
    # Train loop
    # -------------------------
    model.train()
    train_it = iter(train_dl)

    max_steps = int(max_steps)
    eval_every = int(eval_every)
    val_max_batches = int(val_max_batches) if val_max_batches is not None else None




    
    
    X = next(train_it)
    
    # X shape should be (B, 1024, 15) and integer type
    print("dtype", X.dtype, "shape", X.shape)
    
    # Basic stats
    f0 = X[:, :, 0]
    f4 = X[:, :, 4]
    
    print("f0 min/max:", f0.min().item(), f0.max().item())
    print("f4 min/max:", f4.min().item(), f4.max().item())
    
    # If your model uses 32/32/16 bins:
    MAX_ORDER_INDEX = 3 * 32 * 32 * 16  # 49152
    assert (f0 >= 0).all() and (f0 < MAX_ORDER_INDEX).all(), "order_index out of range"
    
    # If f4 is "seconds since open" it should be within a day (paper uses intraday notion)
    # In your earlier code you clipped to [0, 23399]
    assert (f4 >= 0).all() and (f4 <= 23399).all(), "time feature out of range (did you forget ns->sec?)"



    """
    pbar = tqdm(range(1, max_steps + 1), desc=f"train ({model_variant}, frac={train_fraction})")
    for step in pbar:
        X = next(train_it)
        X = X.to(device, non_blocking=True)

        opt.zero_grad(set_to_none=True)

        with torch.amp.autocast("cuda", enabled=(USE_AMP and device.type == "cuda"), dtype=AMP_DTYPE):
            logits = model(X)
            loss = lm_loss_all_positions(logits, X)

        if use_scaler:
            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()
        else:
            loss.backward()
            opt.step()

        pbar.set_postfix(train_loss=f"{loss.item():.4f}", params=f"{n_params/1e6:.2f}M")

        if step % eval_every == 0:
            val_loss = compute_val_loss(model, val_dl, val_max_batches)
            print(f"\nstep {step:6d} | val_loss {val_loss:.4f} | params {n_params/1e6:.2f}M\n")
            model.train()

    """

if __name__ == "__main__":
    train_fn()


dtype torch.int64 shape torch.Size([4096, 1024, 15])
f0 min/max: 6016 49144
f4 min/max: 19800 43198


AssertionError: time feature out of range (did you forget ns->sec?)